In [1]:
import cv2
import pytesseract
import pandas as pd
import numpy as np
import re
import os
import glob

In [2]:
def extract_invoice_data(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Failed to load image: {image_path}")
        return {}
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    text = pytesseract.image_to_string(gray)
    
    invoice_no = re.search(r"Invoice no:\s*(\d+)", text)
    invoice_no = invoice_no.group(1) if invoice_no else ""
    
    date = re.search(r"Date of issue:\s*([0-9/]+)", text)
    date = date.group(1) if date else ""
    
    seller = re.search(r"Seller:\n([\s\S]*?)\n\n", text)
    seller = seller.group(1).strip() if seller else ""
    
    client = re.search(r"Client:\n([\s\S]*?)\n\n", text)
    client = client.group(1).strip() if client else ""
    
    seller_tax = re.search(r"Tax Id:\s*([0-9\-]+)", text)
    seller_tax = seller_tax.group(1) if seller_tax else ""
    
    iban = re.search(r"IBAN:\s*([A-Z0-9]+)", text)
    iban = iban.group(1) if iban else ""
    
    client_tax = re.findall(r"Tax Id:\s*([0-9\-]+)", text)
    client_tax = client_tax[1] if len(client_tax) > 1 else ""
    
    return {
        "invoice_no": invoice_no,
        "date": date,
        "seller": seller,
        "client": client,
        "seller_tax": seller_tax,
        "iban": iban,
        "client_tax": client_tax
    }

In [3]:
invoice_folder = "invoices"
image_files = glob.glob(os.path.join(invoice_folder, "*.jpg"))
data = []

In [4]:
for image_file in image_files:
    data.append(extract_invoice_data(image_file))

In [5]:
df = pd.DataFrame(data)

In [6]:
df.to_csv("extracted_invoices.csv", index=False)
print("Data extracted and saved to extracted_invoices.csv")

Data extracted and saved to extracted_invoices.csv
